# 04 — Pipeline Evaluation: Đo Hit Rate & Latency

**Vai trò:** Pipeline Engineer · **Task:** S4-PE-06 (Yêu cầu 9.7)

Notebook này định nghĩa một **tập câu hỏi đánh giá** (evaluation set) với từ khoá kỳ vọng, chạy từng câu qua `RAGPipeline.query()` (S3-PE-03) thật, rồi tính **Hit Rate** (tỉ lệ từ khoá xuất hiện trong câu trả lời) và **Latency trung bình** — đúng theo cấu trúc cell mẫu design.md §2.5. Đây là cách đo lường khách quan để biết một cấu hình pipeline (model, chunk size, top-k...) có đang hoạt động tốt hay không, thay vì chỉ "cảm nhận" qua vài lần thử thủ công.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.loader import DocumentLoader
from src.data.chunker import TextChunker
from src.embeddings.embedding_model import OllamaEmbeddingModel
from src.embeddings.vector_store import ChromaVectorStore
from src.generation.llm_client import OllamaClient
from src.generation.prompt_builder import PromptBuilder
from src.models import ChunkStrategy, RAGResponse
from src.pipeline.experiment_tracker import ExperimentTracker
from src.pipeline.rag_pipeline import RAGPipeline

pipeline = RAGPipeline(
    loader=DocumentLoader(),
    chunker=TextChunker(strategy=ChunkStrategy.RECURSIVE, chunk_size=300, chunk_overlap=50),
    embedding_model=OllamaEmbeddingModel(model_name="nomic-embed-text"),
    vector_store=ChromaVectorStore(collection_name="notebook_pipeline_evaluation", persist_dir=None),
    llm_client=OllamaClient(model_name="llama3", max_tokens=200),
    prompt_builder=PromptBuilder(),
    top_k=3,
    experiment_tracker=ExperimentTracker(),
)
print(f"Project root: {PROJECT_ROOT}")
print("RAGPipeline da san sang — top_k =", pipeline.top_k)

## 1. Chuẩn bị: index tài liệu mẫu để có ngữ cảnh cho đánh giá

Tập câu hỏi đánh giá ở mục 2 hỏi về RAG và ChromaDB — notebook tạo và index (idempotent, Yêu cầu 9.1) hai tài liệu mẫu phủ đúng hai chủ đề đó, để `pipeline.query()` có ngữ cảnh thật để truy xuất và trả lời.

In [ ]:
RAW_DIR = PROJECT_ROOT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

eval_corpus = {
    "eval_corpus_rag.txt": (
        "Retrieval-Augmented Generation (RAG) la kien truc ket hop retrieval va "
        "generation: he thong tim cac doan van ban lien quan tu kho du lieu rieng "
        "truoc khi yeu cau LLM sinh cau tra loi, giup giam hien tuong ao giac va "
        "bam sat nguon tai lieu thuc te. RAG gom hai giai doan chinh: indexing "
        "(tai tai lieu, chia chunk, tao embedding, luu vao vector store) va "
        "querying (nhung cau hoi, truy xuat ngu canh, sinh cau tra loi)."
    ),
    "eval_corpus_chromadb.txt": (
        "ChromaDB la mot vector database dung de luu tru cac embedding vector "
        "cung voi metadata cua tung chunk. Du lieu duoc luu duoi dang vector so "
        "nhieu chieu, cho phep tim kiem tuong dong (similarity search) bang cach "
        "do khoang cach/diem cosine giua vector cau hoi va vector cac chunk da "
        "luu. ChromaDB ho tro ca che do in-memory (thu nghiem nhanh trong "
        "notebook) va persistent (luu tru lau dai tren dia cho dashboard)."
    ),
}

available = pipeline.llm_client.is_available()
print(f"OLLAMA kha dung tai {pipeline.llm_client.base_url}: {available}")

indexing_results = []
if available:
    for name, content in eval_corpus.items():
        path = RAW_DIR / name
        if not path.exists():
            path.write_text(content, encoding="utf-8")
        result = pipeline.index_document(str(path))
        indexing_results.append(result)
        print(f"  index({name}) -> success={result.success}  num_chunks={result.num_chunks}")
        assert result.success, result.error_message
else:
    print(
        "\n⚠️  OLLAMA chua san sang — bo qua indexing thuc. Hay chay 'ollama serve' "
        "va 'ollama pull llama3' / 'ollama pull nomic-embed-text' roi chay lai notebook."
    )

## 2. Tập câu hỏi đánh giá (evaluation set)

Mỗi mục gồm một câu hỏi và danh sách `expected_keywords` — các từ khoá mà một câu trả lời "tốt" được kỳ vọng nhắc tới (cấu trúc đúng theo design.md §2.5).

In [ ]:
eval_questions = [
    {"question": "RAG la gi?", "expected_keywords": ["retrieval", "generation"]},
    {"question": "ChromaDB luu tru du lieu nhu the nao?", "expected_keywords": ["vector"]},
    {"question": "RAG gom nhung giai doan nao?", "expected_keywords": ["indexing", "querying"]},
]
print(f"So cau hoi danh gia: {len(eval_questions)}")
for item in eval_questions:
    print(f"  - {item['question']!r}  (tu khoa ky vong: {item['expected_keywords']})")

## 3. Hàm tính Hit Rate

Theo đúng pseudocode design.md §2.5: tỉ lệ phần trăm từ khoá kỳ vọng xuất hiện (không phân biệt hoa/thường) trong `response.answer`.

**Preconditions:** `response.answer` không rỗng; `expected_keywords` là `list[str]` không rỗng.
**Postconditions:** trả về `float` trong `[0.0, 1.0]` — `1.0` nghĩa là tất cả từ khoá đều xuất hiện.

In [ ]:
def hit_rate(response: RAGResponse, expected_keywords: list) -> float:
    assert response.answer, "response.answer khong duoc rong"
    assert expected_keywords, "expected_keywords khong duoc rong"

    answer_lower = response.answer.lower()
    hits = sum(1 for kw in expected_keywords if kw.lower() in answer_lower)
    score = hits / len(expected_keywords)

    assert 0.0 <= score <= 1.0
    return score

## 4. Chạy đánh giá — gọi `pipeline.query()` cho từng câu hỏi

Mỗi câu hỏi được hỏi qua `RAGPipeline.query()` thật (Embed → Retrieve → Build Prompt → Generate, pseudocode §2.7). `RAGResponse.latency_ms` đo trọn vẹn cả 4 bước (Yêu cầu 7.4); `ExperimentTracker` đã được gắn vào `pipeline` (S4-PE-04) nên mỗi lượt query cũng tự động được ghi lại — không cần gọi `log_query()` thủ công.

In [ ]:
results = []
if available and indexing_results and all(r.success for r in indexing_results):
    for item in eval_questions:
        response = pipeline.query(item["question"])
        score = hit_rate(response, item["expected_keywords"])
        results.append({
            "question": item["question"],
            "score": score,
            "latency_ms": response.latency_ms,
            "num_contexts": len(response.contexts),
        })
        print(
            f"  [{score:.0%}] {item['question']!r} "
            f"-> {response.latency_ms:.0f} ms, {len(response.contexts)} ngu canh"
        )
else:
    print("Bo qua chay danh gia thuc — xem canh bao o muc 1 (OLLAMA/indexing chua san sang).")

## 5. Tổng kết — Hit Rate & Latency trung bình

Gộp kết quả của toàn bộ tập đánh giá thành hai chỉ số tổng quan: **Hit Rate trung bình** (chất lượng trả lời theo từ khoá kỳ vọng) và **Latency trung bình** (tốc độ xử lý) — đúng yêu cầu của notebook này (Yêu cầu 9.7).

In [ ]:
if results:
    avg_hit_rate = sum(r["score"] for r in results) / len(results)
    avg_latency = sum(r["latency_ms"] for r in results) / len(results)

    print(f"Hit Rate trung binh : {avg_hit_rate:.2%}")
    print(f"Latency trung binh  : {avg_latency:.0f} ms")

    assert 0.0 <= avg_hit_rate <= 1.0
    assert avg_latency > 0

    print("\nTom tat phien thuc nghiem (ExperimentTracker — tu dong ghi qua S4-PE-04):")
    print(pipeline.experiment_tracker.get_summary())
else:
    print("Khong co ket qua de tong hop — xem canh bao o cac muc truoc.")

## 6. Tổng kết

- `hit_rate()` lượng hoá chất lượng câu trả lời theo tỉ lệ từ khoá kỳ vọng xuất hiện — một proxy đơn giản nhưng hữu ích để so sánh khách quan giữa các cấu hình pipeline (đổi `chunk_size`, `top_k`, model...) thay vì chỉ đánh giá "cảm tính" qua vài lần thử thủ công.
- `RAGResponse.latency_ms` cho biết tốc độ xử lý thực tế của toàn luồng `query()` — kết hợp với Hit Rate, đây chính là cặp chỉ số **chất lượng × tốc độ** mà người học RAG cần cân bằng khi tinh chỉnh hệ thống.
- Vì `pipeline` được khởi tạo với `experiment_tracker=ExperimentTracker()` (S4-PE-04), mọi lượt `index_document()`/`query()` ở trên đã **tự động** được ghi lại — đúng tinh thần Yêu cầu 9.8 ("ExperimentTracker SHALL có thể ghi lại kết quả thực nghiệm... mà không làm gián đoạn luồng notebook"). Có thể `save_session()` kết quả này và đối chiếu với các phiên khác (`compare_sessions()`) ngay trên trang **Experiment Log** của dashboard.